# Use Case 6: Transfer Learning using MobileNetV2

This notebook reuses a model pretrained on ImageNet and replaces its original classifier.

Each code cell is preceded by Markdown that explains what the step does, why it is needed, and which deep-learning concept is being demonstrated.


## Step 1: Import libraries

MobileNetV2 and its preprocessing function are imported.

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras import Sequential
from tensorflow.keras.layers import (
    Input,
    Rescaling,
    Conv2D,
    MaxPooling2D,
    Flatten,
    Dense,
    Dropout,
    GlobalAveragePooling2D,
)

from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input


## Step 2: Configure and load images

Images are resized to 160 × 160 for MobileNetV2.

In [ ]:
DATASET_PATH = "../datasets/02_multiclass_objects"
IMAGE_SIZE = (160, 160)
BATCH_SIZE = 16

DATASET_DIR = Path(DATASET_PATH)

train_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
)

class_names = train_ds.class_names

print("Classes:", class_names)
print("Training batches:", len(train_ds))
print("Validation batches:", len(val_ds))


## Step 3: Load the pretrained feature extractor

`include_top=False` removes the original classifier. The first run may download ImageNet weights.

In [ ]:
base_model = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(160, 160, 3),
)

base_model.trainable = False

print("Layers in base model:", len(base_model.layers))


## Step 4: Add a new classifier

The frozen pretrained model extracts features. New layers learn the custom classes.

In [ ]:
inputs = tf.keras.Input(shape=(160, 160, 3))
x = preprocess_input(inputs)
x = base_model(x, training=False)
x = GlobalAveragePooling2D()(x)
x = Dropout(0.3)(x)
outputs = Dense(len(class_names), activation="softmax")(x)

model = tf.keras.Model(inputs, outputs)
model.summary()


## Step 5: Compile and train

Only the new classification layers are trained.

In [ ]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=6,
)


## Step 6: Evaluate

Transfer learning often converges quickly on small datasets.

In [ ]:
loss, accuracy = model.evaluate(val_ds)

print("Validation loss:", round(loss, 4))
print("Validation accuracy:", round(accuracy, 4))

plt.figure(figsize=(8, 4))
plt.plot(history.history["accuracy"], label="Training Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Accuracy Curve")
plt.legend()
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss Curve")
plt.legend()
plt.show()
